# Week 8: Max-Heaps, Priority Queues & Hash Tables

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

O(n) Floyd heapify, Heapsort, and Chaining vs. Linear Probing probe benchmarks.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-08/labs/W08_Heaps_PriorityQueues_HashTables.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Binary Max-Heap, $O(n)$ Floyd Heapify & Heapsort

A complete binary tree fits inside a flat Python list with `parent = (i - 1) // 2`, `left = 2*i + 1`, and `right = 2*i + 2`. Bottom-up `heapify` runs in $O(n)$ because half the nodes are leaves.

In [1]:
import random


class MaxHeap:
    def __init__(self, items=None):
        self._a = list(items) if items else []
        for i in range(len(self._a) // 2 - 1, -1, -1):
            self._sift_down(i)

    def __len__(self):
        return len(self._a)

    def push(self, val):
        self._a.append(val)
        i = len(self._a) - 1
        while i > 0:
            p = (i - 1) // 2
            if self._a[i] > self._a[p]:
                self._a[i], self._a[p] = self._a[p], self._a[i]
                i = p
            else:
                break

    def pop(self):
        top = self._a[0]
        last = self._a.pop()
        if self._a:
            self._a[0] = last
            self._sift_down(0)
        return top

    def _sift_down(self, i):
        n = len(self._a)
        while True:
            l, r, big = 2 * i + 1, 2 * i + 2, i
            if l < n and self._a[l] > self._a[big]: big = l
            if r < n and self._a[r] > self._a[big]: big = r
            if big == i:
                break
            self._a[i], self._a[big] = self._a[big], self._a[i]
            i = big


rng = random.Random(8)
vals = [rng.randint(1, 999) for _ in range(50)]
h = MaxHeap(vals)
desc = [h.pop() for _ in range(len(vals))]
assert desc == sorted(vals, reverse=True)
print("Top 5 extracted:", desc[:5])


Top 5 extracted: [986, 934, 924, 919, 851]


## 2. Hash Tables: Separate Chaining vs. Linear Probing Load-Factor Benchmark

As load factor $\alpha = n / m$ rises toward $1.0$, open-addressing linear probing probe chains grow much faster than chaining bucket lengths due to primary clustering.

In [2]:
TOMB = object()


class LinearProbingMap:
    def __init__(self, cap=256):
        self.k = [None] * cap
        self.v = [None] * cap
        self.n = 0

    def put(self, key, val):
        cap = len(self.k)
        idx = hash(key) % cap
        probes = 0
        while True:
            probes += 1
            if self.k[idx] is None or self.k[idx] is TOMB or self.k[idx] == key:
                if self.k[idx] is None or self.k[idx] is TOMB:
                    self.n += 1
                self.k[idx], self.v[idx] = key, val
                return probes
            idx = (idx + 1) % cap


class ChainingMap:
    def __init__(self, cap=256):
        self.b = [[] for _ in range(cap)]
        self.n = 0

    def put(self, key, val):
        bucket = self.b[hash(key) % len(self.b)]
        probes = 1 + len(bucket)
        for i, (ek, _) in enumerate(bucket):
            if ek == key:
                bucket[i] = (key, val)
                return probes
        bucket.append((key, val))
        self.n += 1
        return probes


for alpha_pct in (25, 50, 75, 90):
    m = 256
    count = int(m * alpha_pct / 100)
    lp, ch = LinearProbingMap(m), ChainingMap(m)
    p_lp = sum(lp.put(f"k_{i}", i) for i in range(count)) / count
    p_ch = sum(ch.put(f"k_{i}", i) for i in range(count)) / count
    print(f"alpha={alpha_pct/100:.2f} -> avg probes: linear_probing={p_lp:.2f}, chaining={p_ch:.2f}")


alpha=0.25 -> avg probes: linear_probing=1.28, chaining=1.14
alpha=0.50 -> avg probes: linear_probing=1.47, chaining=1.24
alpha=0.75 -> avg probes: linear_probing=1.90, chaining=1.31
alpha=0.90 -> avg probes: linear_probing=3.16, chaining=1.35
